In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [36]:
def get_intersection_for_clip(X, Y, epsilon):
    N = len(X)
    M = len(Y)

    X_np = X.detach().cpu().numpy()
    Y_np = Y.detach().cpu().numpy()

    diff = np.abs(X_np[:, np.newaxis] - Y_np[np.newaxis, :])
    contingency_table = (diff < epsilon).astype(float)

    total_sum = np.sum(contingency_table)
    if total_sum == 0: return 0.0
        
    p_xy = contingency_table / total_sum

    p_x = np.sum(p_xy, axis=1) # X의 주변 확률 분포
    p_y = np.sum(p_xy, axis=0) # Y의 주변 확률 분포
    
    def entropy(p):
        mask = p > 0
        return -np.sum(p[mask] * np.log(p[mask]))

    h_x = entropy(p_x)
    h_y = entropy(p_y)
    h_xy = entropy(p_xy)

    # 4. MI = H(X) + H(Y) - H(XY)
    mutual_info = h_x + h_y - h_xy
    
    return max(0.0, mutual_info)

In [38]:
rna_vec = torch.randn(512)
protein_vec = torch.randn(512)

self_mi_test = rna_vec.clone()

mi_random = get_intersection_for_clip(rna_vec, protein_vec, epsilon=1e-5)
print(f"Random Vectors MI: {mi_random:.4f}")

mi_self = get_intersection_for_clip(rna_vec, self_mi_test, epsilon=1e-5)
print(f"Self-Comparison MI: {mi_self:.4f}")

if mi_self > mi_random:
    print("✅ 테스트 성공: 자기 자신과의 MI가 랜덤 벡터 간의 MI보다 높게 측정되었습니다.")
else:
    print("⚠️ 경고: Epsilon 값이 너무 작거나 커서 변별력이 낮을 수 있습니다.")

Random Vectors MI: 0.0000
Self-Comparison MI: 6.2383
✅ 테스트 성공: 자기 자신과의 MI가 랜덤 벡터 간의 MI보다 높게 측정되었습니다.


In [27]:
# Testing several epsilon values
for eps in [0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.00001, 5e-06, 1e-06]:
    mi_rnd = get_intersection_for_clip(rna_vec, protein_vec, epsilon=eps)
    mi_slf = get_intersection_for_clip(rna_vec, rna_vec, epsilon=eps)
    print(f"[Eps: {eps}] Random: {mi_rnd:.4f} | Self: {mi_slf:.4f}")

[Eps: 0.1] Random: 2.6657 | Self: 2.6363
[Eps: 0.05] Random: 3.3416 | Self: 3.2835
[Eps: 0.01] Random: 4.6179 | Self: 4.6338
[Eps: 0.005] Random: 4.9970 | Self: 5.0584
[Eps: 0.001] Random: 4.6809 | Self: 5.8510
[Eps: 0.0005] Random: 4.2781 | Self: 6.0077
[Eps: 1e-05] Random: 1.3863 | Self: 6.2383
[Eps: 5e-06] Random: 0.0000 | Self: 6.2383
[Eps: 1e-06] Random: 0.0000 | Self: 6.2383
